In [1]:
import numpy as np
import copy

from pydrake.all import (
    StartMeshcat,
    RobotDiagramBuilder,
    RigidTransform,
    RotationMatrix,
    MeshcatVisualizer,
    MeshcatVisualizerParams,
    RobotDiagram,
    Role,
    IrisInConfigurationSpaceFromCliqueCover,
    IrisFromCliqueCoverOptions,
    CollisionCheckerParams,
    SceneGraphCollisionChecker,
    RandomGenerator,
    IrisNp,
    IrisOptions,
    GcsTrajectoryOptimization,
    GraphOfConvexSetsOptions,
    Point,
)

from manipulation.meshcat_utils import PublishPositionTrajectory

In [2]:
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7003


In [3]:
meshcat.Delete()
builder = RobotDiagramBuilder()
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = builder.parser()

mat = parser.AddModelsFromUrl("file:///home/noor/SO-ARM100/Simulation/assets/mat.sdf")[0]
plant.WeldFrames(plant.world_frame(), plant.GetFrameByName("mat_link"))
so101 = parser.AddModelsFromUrl("file:///home/noor/SO-ARM100/Simulation/SO101/so101_new_calib_urdf_drake_hydro.urdf")[0]
plant.WeldFrames(
    plant.GetFrameByName("mat_link"),
    plant.GetFrameByName("base_link"), 
    RigidTransform(
        RotationMatrix.MakeZRotation(np.pi / 2),
        [0, -0.1775, 0.0074]
    )
)
box = parser.AddModelsFromUrl("file:///home/noor/SO-ARM100/Simulation/assets/box.sdf")[0]
plant.WeldFrames(
    plant.GetFrameByName("mat_link"),
    plant.GetFrameByName("box_link"),
    RigidTransform(
        [-0.075, 0.025, 0.02],
    )
)

plant.Finalize()

visualizer = MeshcatVisualizer.AddToBuilder(
    builder.builder(),
    scene_graph,
    meshcat,
    MeshcatVisualizerParams(role=Role.kIllustration),
)

diagram: RobotDiagram = builder.Build()

context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(context)

collision_checker_params = CollisionCheckerParams()
collision_checker_params.model = diagram
collision_checker_params.robot_model_instances = [so101]
collision_checker_params.edge_step_size = 0.01
collision_checker = SceneGraphCollisionChecker(collision_checker_params)

INFO:drake:Allocating contexts to support implicit context parallelism 24


In [4]:
q_rest = np.array([0, -1.822, 1.55, 0.906, 0, 0.0])
q_ready = np.array([0, 0, 0, 1.5, 0, 0])
q_goal = np.array([-0.47976962, -0.23351724, 0.72552294, 1.02322157, -0.40196476, 0.56771817])

options = IrisOptions()
options.num_collision_infeasible_samples = 3
options.random_seed = 1235
options.require_sample_point_is_contained = True

regions = []
for q in [q_rest, q_ready, q_goal]:
    print('computing next region...')
    plant.SetPositions(plant_context, so101, q)
    region = IrisNp(plant, plant_context, options)
    regions.append(region)

INFO:drake:IrisNp iteration 0


computing next region...


INFO:drake:IrisNp iteration 1
INFO:drake:IrisNp iteration 2
INFO:drake:IrisNp iteration 3
INFO:drake:IrisNp: terminating iterations because the seed point is no longer in the region.
INFO:drake:IrisNp iteration 0
INFO:drake:IrisNp iteration 1


computing next region...


INFO:drake:IrisNp iteration 2
INFO:drake:IrisNp iteration 3
INFO:drake:IrisNp iteration 4
INFO:drake:IrisNp iteration 5
INFO:drake:IrisNp: terminating iterations because the seed point is no longer in the region.
INFO:drake:IrisNp iteration 0
INFO:drake:IrisNp: Terminating because the hyperellipsoid volume change 0.011039215914811001 is below the threshold 0.02.


computing next region...


In [5]:
order = 5
continuity_order = 4
trajopt = GcsTrajectoryOptimization(plant.num_positions())
gcs_regions = trajopt.AddRegions(regions, order=order)
source = trajopt.AddRegions([Point(q_rest)], order=0)
target = trajopt.AddRegions([Point(q_goal)], order=0)
trajopt.AddEdges(source, gcs_regions)
trajopt.AddEdges(gcs_regions, target)
trajopt.AddTimeCost()
# trajopt.AddPathLengthCost()
# trajopt.AddPathEnergyCost()
trajopt.AddVelocityBounds(plant.GetVelocityLowerLimits(), plant.GetVelocityUpperLimits())
for o in range(1, continuity_order + 1):
    print(f"adding C{o} constraints")
    trajopt.AddContinuityConstraints(o)
options = GraphOfConvexSetsOptions()
[traj, result] = trajopt.SolvePath(source, target, options)
if result.is_success():
    print('success')
    PublishPositionTrajectory(traj, context, plant, visualizer)
else:
    print('failure')

INFO:drake:Solved GCS shortest path using Clp with convex_relaxation=true and preprocessing=true and rounding.
INFO:drake:Found 2 unique paths, discarded 98 duplicate paths.
INFO:drake:Finished 2 rounding solutions with SNOPT.


adding C1 constraints
adding C2 constraints
adding C3 constraints
adding C4 constraints
success
